# Chat with Qwen3-4B + DSpark speculative decoding

Talks to the local vLLM server started with:

```
CUDA_VISIBLE_DEVICES=0 VLLM_WSL2_ENABLE_PIN_MEMORY=1 uv run vllm serve Qwen/Qwen3-4B \
  --speculative-config '{"method": "dspark", "model": "deepseek-ai/dspark_qwen3_4b_block7", "num_speculative_tokens": 7}' \
  --trust-remote-code --tensor-parallel-size 1 --gpu-memory-utilization 0.87 \
  --max-model-len 2048 --max-num-seqs 4 --enforce-eager
```

Make sure that server is running (check `curl http://localhost:8000/health`) before using this notebook.

Qwen3 reasons by default (emits a `<think>...</think>` block before the actual
answer), which can eat the whole `max_tokens` budget on short limits. Set
`ENABLE_THINKING = True` below if you want to see the reasoning trace.

In [1]:
from openai import OpenAI

client = OpenAI(base_url="http://localhost:8000/v1", api_key="not-needed")
MODEL = "Qwen/Qwen3-4B"
ENABLE_THINKING = False

messages = []

## One-off message

Edit the string below and re-run this cell to send a single message and print the reply. Conversation history is kept in `messages`, so run this cell repeatedly to keep chatting.

In [2]:
user_input = "Hello! What model are you?"

messages.append({"role": "user", "content": user_input})

response = client.chat.completions.create(
    model=MODEL,
    messages=messages,
    temperature=0.7,
    max_tokens=512,
    extra_body={"chat_template_kwargs": {"enable_thinking": ENABLE_THINKING}},
)

reply = response.choices[0].message.content
messages.append({"role": "assistant", "content": reply})
print(reply)

Hello! I am Qwen, a large-scale language model developed by Alibaba Cloud. I can help with a wide range of tasks, including answering questions, creating content, and more. How can I assist you today?


## Interactive loop (optional)

Runs an `input()` loop in the notebook itself. Type `exit` or `quit` to stop.

In [ ]:
while True:
    user_input = input("You: ")
    if user_input.strip().lower() in ("exit", "quit"):
        break

    messages.append({"role": "user", "content": user_input})

    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        temperature=0.7,
        max_tokens=512,
        extra_body={"chat_template_kwargs": {"enable_thinking": ENABLE_THINKING}},
    )

    reply = response.choices[0].message.content
    messages.append({"role": "assistant", "content": reply})
    print(f"Model: {reply}\n")

## Reset conversation

Run this to clear history and start a fresh conversation.

In [4]:
messages = []
print("Conversation cleared.")

Conversation cleared.
